In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
import os
import qnas_config as cfg
from util import check_files
from cnn.input import GenericDataLoader
from cnn.train_detailed import train_and_eval

In [ ]:
phase = 'retrain'
experiment_path = os.path.join("experiments", "exp13_adamw_repeat_1")
config_file = 'config_files/config11.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'retrain_folder': 'retrain',
    'data_path': 'cifar10_data',
    'log_level': 'INFO',
    'max_epochs': 300,
    'epochs_to_eval': 10,
    'batch_size': 256,
    'eval_batch_size': 1000,
    'limit_data': False,
    'num_workers': 4,
}

In [ ]:
check_files(args['experiment_path'])
config = cfg.ConfigParameters(args, phase=phase)
config.get_parameters()

fn_dict=config.fn_dict

In [ ]:
config.load_evolved_data(experiment_path=experiment_path)
params = config.train_spec
params

In [ ]:
evolved_params = config.evolved_params
evolved_params['net']

In [ ]:
data_loader = GenericDataLoader(params=params)

In [ ]:
train_loader, val_loader = data_loader.get_loader(pin_memory_device='cuda:0')
test_loader = data_loader.get_loader(for_train=False, pin_memory_device='cuda:0')

In [ ]:
retrain_multi = []
for i in range(10):
    retrain_multi.append(train_and_eval(params=params, fn_dict=fn_dict, net_list=evolved_params['net'], 
                                train_loader=train_loader, val_loader=val_loader, test_loader=test_loader))

In [ ]:
with open(os.path.join(experiment_path, 'retrain_results.txt'), 'w') as f:
    for item in retrain_multi:
        f.write("%s\n" % item)